# M12 on Kaggle: the adaptation-policy agent on E4 (`e4_added`, D11), full pool

**Seed:** set `SEED` in the first code cell. Run 42 first, then 43 in a second session. M12 reads M9's report for the same seed (K and the scores the none arm must match), which is in the repo.

**Attach as input:**
1. The same private Kaggle dataset as `m3_kaggle.ipynb` (the five files of the M1 `processed/` folder).
2. The model: *Add Input → Models →* **Gemma 2**, *Transformers*, **gemma-2-2b-it** (as in M11).

**Settings:** Internet *On*. Accelerator: **GPU T4**. **Run:** *Save Version → Save & Run All (Commit)*.

Steps: a GPU smoke test of 4 policy decisions, the M2 partition, then three arms, each in its own process: `none` (E4 unchanged, about 80 min), `heuristic` and `llm` (each a full E4 pass plus the decisions). The training runs on the CPU at 4 threads, as in M9; only the llm arm uses the GPU. Every step prints an exit code: 137 means the process was killed (usually out of memory).

**Output:** `/kaggle/working/m12_full_seed42_outputs.zip` (the name follows `SEED`) with `reports/m12/full/` and the run folders. Bring it back, also after a failure.

In [ ]:
SEED = 42
TAG = "" if SEED == 42 else f"seed{SEED}"   # seed 42 is the main report, as in M7
TAG_ARG = f"--tag {TAG}" if TAG else ""
import os, subprocess, sys
REPO = "/kaggle/working/IDS-code"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Indrajit147/IDS-code.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
# --no-deps: keep Kaggle's preinstalled stack intact; versions are recorded per run. bitsandbytes isn't preinstalled.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO, "--no-deps"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"], check=True)
os.chdir(REPO)
sys.path.insert(0, f"{REPO}/src")   # this kernel started before the editable install, so it cannot see it
os.environ["IDS_REPO_ROOT"] = REPO
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
assert os.path.exists("scripts/m12_policy.py"), "This clone predates M12 (no scripts/m12_policy.py). Restart the session so it clones the latest main."

In [ ]:
# The model: a Kaggle model input if attached, else Hugging Face (needs the HF_TOKEN secret)
import glob
found = sorted(os.path.dirname(p) for p in glob.glob("/kaggle/input/**/config.json", recursive=True) if "gemma-2-2b-it" in p.lower())
if found:
    MODEL = found[-1]
else:
    MODEL = "google/gemma-2-2b-it"
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("model:", MODEL)
LLM_SET = f"policy.backend=hf llm.model_id={MODEL}"

In [ ]:
# GPU smoke test: 4 policy decisions on made-up evidence. Stops here on failure.
import json, time, torch
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4"
from ids.llm.policy import decide
from ids.llm.verifier import HFBackend
be = HFBackend(MODEL, "4bit", "float16", 256)
EV = {"adwin_fired": True, "fired_at_round_share": 0.3, "rows_new": 9000, "error_rate_new_rows": 0.21, "error_rate_validation": 0.05,
      "new_classes": ["recon"], "class_share_new_rows": {"dos": 0.6, "recon": 0.4}, "class_share_buffer": {"benign": 0.3, "dos": 0.7},
      "per_class_f1_validation": {"benign": 0.99, "dos": 0.97}, "feature_shift_share": 0.3, "most_shifted_features": ["Rate", "syn_count", "AVG", "Std", "UDP"],
      "shap_drift": 0.42, "top_shap_features_new_rows": ["Rate", "Header_Length", "syn_count", "AVG", "Number"], "previous_policy": None}
tic = time.perf_counter()
recs = [decide(EV, be, 2) for _ in range(4)]
sec = time.perf_counter() - tic
print(json.dumps(recs[0]["policy"], indent=1))
print("valid:", sum(r["valid"] for r in recs), "of 4; first try:", sum(r["first_try_valid"] for r in recs), f"; {sec / 4:.1f} s per decision")
del be; torch.cuda.empty_cache()
assert sum(r["valid"] for r in recs) >= 3, "The model's replies mostly fail the contract: stop and bring the output back"

In [ ]:
# Rebuild the M2 partition for the full pool (deterministic, ~1 min)
!python scripts/m2_build_partitions.py --env kaggle --pools full
# M2 redraws these figures, and their bytes differ on Kaggle. Nothing reads them. A changed report or JSON stays flagged.
!git checkout -- 'reports/m2/full/*.png'

In [ ]:
import glob, os, shutil
REP = os.path.join("reports/m12/full", TAG)
def pack():
    stage = "/kaggle/working/m12_stage"
    shutil.rmtree(stage, ignore_errors=True)
    os.makedirs(stage)
    if os.path.isdir(REP):
        shutil.copytree(REP, f"{stage}/{REP}")
    for r in sorted(glob.glob("/kaggle/working/results/runs/*_m12_policy_full")):
        shutil.copytree(r, f"{stage}/results/runs/{os.path.basename(r)}")
    shutil.make_archive(f"/kaggle/working/m12_full_seed{SEED}_outputs", "zip", stage)
    shutil.rmtree(stage)
    print("zip written:", sorted(os.listdir(REP)) if os.path.isdir(REP) else [])

In [ ]:
# the three arms, each in its own process (a failure keeps the arms already done), then the report
for arm in ["none", "heuristic", "llm"]:
    print("=====", arm, flush=True)
    !python -X faulthandler scripts/m12_policy.py --env kaggle --pool full --set resources.n_jobs=4 seed={SEED} {LLM_SET} {TAG_ARG} --only {arm}; echo "exit code: $?"
    pack()
!python scripts/m12_policy.py --env kaggle --pool full --set resources.n_jobs=4 seed={SEED} {LLM_SET} {TAG_ARG} --report; echo "exit code: $?"
pack()
if os.path.exists(f"{REP}/POLICY.md"):
    print(open(f"{REP}/POLICY.md", encoding="utf-8").read())